# Solución P01: Perfil y lectura por bloques

[Enunciado](../talleres/P01.md) · [Índice](README.md)

Ejecutar de arriba abajo con el kernel del entorno `.venv` preparado en la [guía WSL](../docs/instalacion-wsl.md). Usa las tres muestras incluidas; no descarga los archivos completos. Las salidas propias se regeneran al repetir la ejecución.

In [ ]:
CODIGO = "P01"
from pathlib import Path
from types import SimpleNamespace
import sys, json
import pandas as pd
from IPython.display import display

# Funciona desde soluciones/, la raíz del repositorio o la carpeta base bigdata.
roots = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'big-data-postgraduate']
REPO = next((p for p in roots if (p / 'kit/pqrs_talleres.py').is_file()), None)
assert REPO is not None, 'Abrir Jupyter desde bigdata/big-data-postgraduate o soluciones/'
KIT = REPO / 'kit'
sys.path.insert(0, str(KIT))
import pqrs_talleres as lab
OUT = KIT / 'salidas/soluciones' / CODIGO
OUT.mkdir(parents=True, exist_ok=True)
a = SimpleNamespace(datos=None, completo=False, bloque=500, out=OUT, repeticiones=3)
print('Repositorio:', REPO, '\nSalidas:', OUT)


## Tarea 1. Manifiesto y corte
El corte del archivo identifica la publicación; `periodo` y `mes` describen cada reporte. No se debe sustituir el año de la fila por el nombre del archivo.

In [ ]:
manifest = lab.fuentes()
display(pd.DataFrame([{k:f[k] for k in ['periodo','separador','archivo','sha256_muestra']} for f in manifest]))

## Tareas 2 y 3. Perfil y comparación de bloques
Se verifica el hash de cada muestra y su esquema. Se conserva cada medición antes de que la siguiente reemplace el perfil.

In [ ]:
mediciones=[]
referencia=None
for bloque in [100,500,1000]:
    a.bloque=bloque
    perfil=lab.perfil(a)
    lab.guardar(OUT/f'perfil_bloque_{bloque}.json',perfil)
    conteos=[(x['filas'],x['vacios'],x['meses']) for x in perfil]
    if referencia is None: referencia=conteos
    assert conteos==referencia
    for x in perfil:
        assert x['columnas']==38 and x['filas']==1000
        mediciones.append({'bloque':bloque, **{k:x[k] for k in ['corte','filas','bytes','memoria_dataframe_bloque_max_bytes']}})
display(pd.DataFrame(mediciones))
display(pd.DataFrame(perfil).set_index('corte')['vacios'].apply(pd.Series).T)

**Respuesta:** los conteos no cambian con el bloque. La memoria medida es la del DataFrame de un bloque, no la RSS del proceso. El tamaño del CSV en disco no permite deducir directamente la RAM necesaria.

## Tarea 4. Diccionario y claves
|Campo|Interpretación de trabajo|
|---|---|
|periodo|Año declarado en el reporte; validar conversión|
|mes|Mes declarado; validar rango 1–12|
|pet_cod_mpio|Municipio del peticionario|
|afec_cod_mpio|Municipio del afectado; no sustituye al del peticionario|
|OBJECTID|Identificador candidato dentro del corte, no prueba de identidad global|

`id_afec` describe al afectado y no se usa como clave de reclamo. La pareja corte–OBJECTID sigue siendo una clave candidata, no una garantía semántica.

In [ ]:
frames=[]
for f,p in lab.seleccion(a):
    d=pd.concat(lab.leer(f,p,500),ignore_index=True)
    d['corte_archivo']=f['periodo']; frames.append(d)
datos=pd.concat(frames,ignore_index=True)
display(datos.groupby(['corte_archivo','periodo','mes']).size().rename('filas').reset_index())
display(datos[['pet_cod_mpio','afec_cod_mpio']].apply(lambda s: s.str.strip().eq('').sum()).rename('vacios'))
print('Parejas corte–OBJECTID repetidas:',datos.duplicated(['corte_archivo','OBJECTID']).sum())

## Tarea 5. Controles y presupuesto para completos
En completos se deben verificar SHA-256, separador, 38 columnas, anchura de cada fila y conteo por corte; después revisar períodos y claves. Reservar al menos el volumen de los originales más productos y temporales, medir RAM con bloques pequeños y ajustarlos según evidencia.

**Límites de cobertura:** las primeras 1.000 filas de cada corte no forman una muestra aleatoria; los reportes PQRS no representan prevalencia clínica. Los perfiles de las muestras no autorizan extrapolar porcentajes a todos los reportes. Los JSON por bloque y las tablas anteriores constituyen la evidencia de entrega.